# Feature engineering week 7 task 1
### Titanic passenger Dataset Generation main goal to predict survived 


In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Load the real Titanic dataset directly from a raw web link
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print("Dataset Shape:", df.shape)
print("\nMissing values per column:\n", df.isnull().sum())
df.head()


Dataset Shape: (891, 12)

Missing values per column:
 PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### Step 1: Drop Low-Value Columns with details why 

### Decisions & Justification:
* **`PassengerId`** & **`Name`**: Dropped because these are unique individual text values. They have maximum cardinality and do not contribute patterns to machine learning algorithms.
* **`Ticket`**: Dropped because ticket numbers are alphanumeric strings with arbitrary naming conventions that lead to high variance without predictive power.
* **`Cabin`**: Dropped because it is missing over 70% of its total data points. Imputing a category with such a high volume of null instances creates artificial bias.


In [5]:
# Dropping the uninformative high-cardinality and mostly null columns
columns_to_drop = ["PassengerId", "Name", "Ticket", "Cabin"]
df_cleaned = df.drop(columns=columns_to_drop)

print("Remaining columns after drop:", df_cleaned.columns.tolist())


Remaining columns after drop: ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']


### Step 2: Impute Missing Values Using 2 Strategies

### Decisions & Justification:
* **Strategy 1 (Median Imputation)**: Applied to the numerical column `Age`. Passenger ages are slightly skewed; the median provides a balanced midpoint immune to extreme outliers.
* **Strategy 2 (Mode/Most Frequent Imputation)**: Applied to the categorical column `Embarked`. Since it is an object/text column representing embarkation ports, we substitute missing fields with the most frequently occurring port ('S').


In [6]:
# Strategy 1: Impute numerical 'Age' with its median
df_cleaned["Age"] = df_cleaned["Age"].fillna(df_cleaned["Age"].median())

# Strategy 2: Impute categorical 'Embarked' with its mode (most frequent item)
most_frequent_port = df_cleaned["Embarked"].mode()[0]
df_cleaned["Embarked"] = df_cleaned["Embarked"].fillna(most_frequent_port)

print("Missing values after pipeline imputation:\n", df_cleaned.isnull().sum())


Missing values after pipeline imputation:
 Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64


### Step 3: Encode Categorical Variables Appropriately

### Decisions & Justification:
* **One-Hot Encoding**: Applied to `Sex` and `Embarked`. These represent nominal classes without mathematical ranking. We leverage `drop='first'` to avoid collinear dependencies (the dummy variable trap).


In [7]:
categorical_cols = ["Sex", "Embarked"]

# Configure and apply One-Hot Encoder
encoder = OneHotEncoder(sparse_output=False, drop="first")
encoded_array = encoder.fit_transform(df_cleaned[categorical_cols])

# Format into a clean Pandas DataFrame with structural column names
encoded_df = pd.DataFrame(
    encoded_array, columns=encoder.get_feature_names_out(categorical_cols)
)

# Merge back with numerical columns
df_numeric = df_cleaned.drop(columns=categorical_cols)
df_final = pd.concat([df_numeric, encoded_df], axis=1)

df_final.head()


,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,0,3,22.0,1,0,7.2500,1.0,0.0,1.0
1,1,1,38.0,1,0,71.2833,0.0,0.0,0.0
2,1,3,26.0,0,0,7.9250,0.0,0.0,1.0
3,1,1,35.0,1,0,53.1000,0.0,0.0,1.0
4,0,3,35.0,0,0,8.0500,1.0,0.0,1.0


### Step 4: Scale Numerics with StandardScaler

### Decisions & Justification:
* **StandardScaler**: Applied to continuous elements `Age` and `Fare`. While `Age` ranges primarily from 0 to 80, `Fare` numbers scale dramatically higher. Standardizing values ensures models don't disproportionately favor larger numeric magnitudes.


In [8]:
numeric_features = ["Age", "Fare"]

scaler = StandardScaler()
df_final[numeric_features] = scaler.fit_transform(df_final[numeric_features])

df_final.head()


,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,0,3,-0.565736,1,0,-0.502445,1.0,0.0,1.0
1,1,1,0.663861,1,0,0.786845,0.0,0.0,0.0
2,1,3,-0.258337,0,0,-0.488854,0.0,0.0,1.0
3,1,1,0.433312,1,0,0.420730,0.0,0.0,1.0
4,0,3,0.433312,0,0,-0.486337,1.0,0.0,1.0


### Step 5: Perform an 80/20 Train-Test Split

### Decisions & Justification:
* We split features (`X`) from our target outcome label (`y` = `Survived`), routing **80% into training operations** and **20% toward blind validation tests**. Random seed state is locked for consistency across model adjustments.


In [9]:
# Separate features from target labels
X = df_final.drop(columns=["Survived"])
y = df_final["Survived"]

# Divide sets into an 80/20 layout split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Output final confirmation metrics
print(f"X_train Matrix Dimensions: {X_train.shape}")
print(f"X_test Matrix Dimensions:  {X_test.shape}")
print(f"y_train Array Vector Size: {y_train.shape}")
print(f"y_test Array Vector Size:  {y_test.shape}")


X_train Matrix Dimensions: (712, 8)
X_test Matrix Dimensions:  (179, 8)
y_train Array Vector Size: (712,)
y_test Array Vector Size:  (179,)
